In [ ]:
globalThis.fs = globalThis.fs || require('fs');
globalThis.Builder = globalThis.Builder || require('selenium-webdriver').Builder;
globalThis.By = globalThis.By || require('selenium-webdriver').By;
globalThis.robotsParser = globalThis.robotsParser || require('robots-parser');
globalThis.fetch = globalThis.fetch || ((...args) => import('node-fetch').then(({default: fetch}) => fetch(...args)));
globalThis.urls = globalThis.urls || [
  'https://ubc.perfectmind.com/24063/Clients/BookMe4LandingPages/Facility?facilityId=c0668c1c-1fd6-4432-a20e-4c50aaad5baa',
  'https://ubc.perfectmind.com/24063/Clients/BookMe4LandingPages/Facility?facilityId=e2d99dda-cdc4-4af4-8df6-6c8061ffd56f'
];

const fs = globalThis.fs;
const Builder = globalThis.Builder;
const By = globalThis.By;
const robotsParser = globalThis.robotsParser;
const fetch = globalThis.fetch;
const urls = globalThis.urls;

async function scrapePage(driver, url, outputFile) {
  console.log(`Scraping: ${url}`);
  await driver.get(url);

  // 隐藏 Selenium webdriver 特征
  await driver.executeScript('Object.defineProperty(navigator, "webdriver", {get: () => undefined})');

  // 随机等待 4-7 秒
  const waitTime = 4000 + Math.floor(Math.random() * 3000);
  await driver.sleep(waitTime);

  const pageSource = await driver.getPageSource();
  fs.writeFileSync(outputFile, pageSource);
  console.log(`Saved to ${outputFile}`);
}

// 修改 checkRobots，声明友好爬虫
async function checkRobots(url) {
  const domain = new URL(url).origin;
  const robotsUrl = domain + '/robots.txt';
  const res = await fetch(robotsUrl, {
    headers: {
      'User-Agent': 'my-node-scraper (friendly bot; contact: your@email.com)'
    }
  });
  const text = await res.text();
  const robots = robotsParser(robotsUrl, text);
  const allowed = robots.isAllowed(url, 'my-node-scraper');
  console.log(`Robots check for ${url}: ${allowed ? 'allowed' : 'disallowed'}`);
  return allowed;
}

function getFacilityId(url) {
  const match = url.match(/facilityId=([a-z0-9-]+)/i);
  return match ? match[1] : 'unknown';
}

(async function main() {
  const startTime = Date.now(); // 记录开始时间
  let driver = await new Builder().forBrowser('chrome').build();

  try {
    for (let i = 0; i < urls.length; i++) {
      const url = urls[i];
      const allowed = await checkRobots(url);
      if (!allowed) {
        console.log(`Skipping ${url} due to robots.txt restrictions.`);
        continue;
      }

      const facilityId = getFacilityId(url);
      const outputFile = `output_${facilityId}.html`;
      await scrapePage(driver, url, outputFile);

      console.log('Waiting 10 seconds before next request...');
      await new Promise(res => setTimeout(res, 10000)); // 避免太快，等10秒
    }
  } catch (err) {
    console.error('Error:', err);
  } finally {
    await driver.quit();
    const endTime = Date.now(); // 记录结束时间
    const duration = ((endTime - startTime) / 1000).toFixed(2);
    console.log(`总耗时: ${duration} 秒`);
  }
})();

Promise { <pending> }

Robots check for https://ubc.perfectmind.com/24063/Clients/BookMe4LandingPages/Facility?facilityId=c0668c1c-1fd6-4432-a20e-4c50aaad5baa: allowed
Scraping: https://ubc.perfectmind.com/24063/Clients/BookMe4LandingPages/Facility?facilityId=c0668c1c-1fd6-4432-a20e-4c50aaad5baa
Saved to output_c0668c1c-1fd6-4432-a20e-4c50aaad5baa.html
Waiting 10 seconds before next request...
Robots check for https://ubc.perfectmind.com/24063/Clients/BookMe4LandingPages/Facility?facilityId=e2d99dda-cdc4-4af4-8df6-6c8061ffd56f: allowed
Scraping: https://ubc.perfectmind.com/24063/Clients/BookMe4LandingPages/Facility?facilityId=e2d99dda-cdc4-4af4-8df6-6c8061ffd56f
Saved to output_e2d99dda-cdc4-4af4-8df6-6c8061ffd56f.html
Waiting 10 seconds before next request...
总耗时: 33.18 秒
